In [61]:
import os
import h5py
import pandas as pd
import numpy as np
import re
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import sys
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore", message="enable_nested_tensor is True")

In [62]:
BASE_DIR = Path("/loctmp/mue37538/dataset/data-transfer-student")

In [63]:
def _get_patch_count(h5_path: Path) -> int:
    with h5py.File(h5_path, "r") as f:
        return f["features"].shape[0]

def build_wsi_cohort(cohort_dir: Path, required_slides: set = None) -> pd.DataFrame:
    df_clin = pd.read_csv(
        cohort_dir / "clinical/clinical_survival_per_slide.csv")
    wsi_dir = cohort_dir / "wsi_features/conch_v15"
    available_wsi = {p.stem for p in wsi_dir.glob("*.h5")}

    df_clin = df_clin.dropna(subset=['survival_months', 'censorship']).copy()
    df_clin = df_clin[df_clin['survival_months'] > 0].copy()
    df_clin['slide_id'] = df_clin['slide_id'].str.replace(
        '.svs', '', regex=False)
    df_clin = df_clin[df_clin["slide_id"].isin(available_wsi)].copy()
    if required_slides is not None:  # nur Slides, die auch Omics-Daten haben
        df_clin = df_clin[df_clin["slide_id"].isin(required_slides)].copy()

    
    df_clin["event"] = 1 - df_clin["censorship"] # 1 = Todesfall, 0 = zensiert


    dx_extracted = df_clin["slide_id"].str.extract(r"-DX(\d+)\.", expand=False)
    df_clin["dx_num"] = dx_extracted.fillna(999).astype(int)

    df_clin["n_patches"] = [
        _get_patch_count(wsi_dir / f"{s}.h5") for s in df_clin["slide_id"]
    ]
    
    df_clin = (
        df_clin.sort_values(by=["case_id", "dx_num", "n_patches"], ascending=[
                            True, True, False])
        .drop_duplicates(subset=["case_id"], keep="first")
        # da wir nur einen Slide pro Patient wollen, wird hier nach case_id gruppiert und der Slide mit der niedrigsten DX-Nummer und den meisten Patches behalten
        .reset_index(drop=True)
    )
    return df_clin

In [64]:
from typing import Tuple

def compute_fold_time_bins(train_df: pd.DataFrame, val_df: pd.DataFrame, n_bins: int = 4) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Berechnet die Quartil-Grenzen NUR auf den Todesfällen von train_df (verhindert Data Leakage)
    und wendet sie auf train_df und val_df an.
    """
    train_df = train_df.copy()
    val_df = val_df.copy()

    # 1. Fit nur auf den unzensierten Trainingsdaten dieses Folds
    uncensored_train = train_df.loc[train_df["event"] == 1, "survival_months"]
    _, q_bins = pd.qcut(uncensored_train, q=n_bins, retbins=True, labels=False)

    # 2. Äußere Grenzen auf +/- unendlich setzen, damit Val-Samples außerhalb
    #    der Trainings-Range sauber in Bin 0 bzw. Bin (n_bins - 1) fallen (keine NaNs!)
    q_bins[0] = -np.inf
    q_bins[-1] = np.inf

    # 3. Transform auf Train und Val mit denselben Trainings-Grenzen
    train_df["time_bin"] = pd.cut(
        train_df["survival_months"], bins=q_bins, labels=False, include_lowest=True
    ).astype(int)
    val_df["time_bin"] = pd.cut(
        val_df["survival_months"], bins=q_bins, labels=False, include_lowest=True
    ).astype(int)

    return train_df, val_df

In [65]:
def cross_validate(df: pd.DataFrame, n_splits: int = 5, n_bins: int = 4, min_events_per_fold: int = 5) -> pd.DataFrame:
    best_split = None
    best_score = float("inf")

    # Berechnet die Zeitbins für die Stratifizierung
    uncensored_times = df.loc[df["event"] == 1, "survival_months"]
    _, q_bins = pd.qcut(uncensored_times, q=n_bins, retbins=True, labels=False)
    q_bins[0] = -np.inf
    q_bins[-1] = np.inf
    strat_bins = pd.cut(df["survival_months"], bins=q_bins, labels=False, include_lowest=True)
    strat_target = np.where(df["event"] == 1, strat_bins + 1, 0)

    # testen von verschiedenen Seeds für die Greedy-Gruppenzuweisung
    for seed in range(1000):
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        
        folds = np.full(len(df), -1)
        valid_split = True
        event_counts = []
        fold_sizes = []

        for fold, (_, val_idx) in enumerate(sgkf.split(X=df, y=strat_target, groups=df["site"])): # fold ist die aktuelle Fold-Nummer, val_idx sind die Indizes der Validierungsdaten für diesen Fold
            folds[val_idx] = fold
            n_events = df.iloc[val_idx]["event"].sum()
            if n_events < min_events_per_fold:
                valid_split = False
                break
            event_counts.append(n_events)
            fold_sizes.append(len(val_idx))

        if valid_split:
            # Optimiere auf möglichst gleiche Anzahl an Events UND ähnliche Fold-Größe
            score = np.std(event_counts) * 2.0 + np.std(fold_sizes) * 0.5
            if score < best_score:
                best_score = score
                best_split = folds.copy()

    if best_split is None:
        raise RuntimeError(
            f"Kein gültiger Split mit mind. {min_events_per_fold} Events pro Fold gefunden! "
            "Eine einzelne 'site' dominiert die Kohorte zu stark."
        )

    df["fold"] = best_split
    
    # Kurzer Check-Print Log:
    print("Fold-Verteilung (Größe | Events):")
    for f in range(n_splits):
        sub = df[df["fold"] == f]
        print(f"  Fold {f}: n={len(sub):3d} | events={int(sub['event'].sum()):2d} | sites={sub['site'].nunique()}")
        
    return df

In [66]:
OMIC_SUFFIXES = ("_rnaseq", "_mut", "_cnv")

def load_omics(cohort_dir: Path) -> pd.DataFrame:
    """Gen-Level RNA / Mutation / CNV (MCAT-Format), eine Zeile pro Slide."""
    omics = pd.read_csv(cohort_dir / "genomics/clinical_omics_per_slide.csv.zip", low_memory=False)
    omics["slide_id"] = omics["slide_id"].str.replace(".svs", "", regex=False)
    gene_cols = [c for c in omics.columns if c.endswith(OMIC_SUFFIXES)]
    return omics[["slide_id"] + gene_cols].drop_duplicates("slide_id")


def get_omic_groups(columns, signatures_csv: Path) -> list:
    """6 MCAT-Signaturen (Tumor Suppressors, Oncogenes, Kinases, ...) -> je eine sortierte Liste vorhandener Spalten."""
    signatures = pd.read_csv(signatures_csv)
    columns = set(columns)
    groups = []
    for col in signatures.columns:
        genes = signatures[col].dropna().unique()
        names = [g + suf for g in genes for suf in OMIC_SUFFIXES]
        groups.append(sorted(n for n in names if n in columns))
    return groups


def build_dataset(cohort_dir: Path, n_bins: int = 4, n_splits: int = 5) -> pd.DataFrame:
    omics = load_omics(cohort_dir)
    df_clin = build_wsi_cohort(cohort_dir, required_slides=set(omics["slide_id"]))
    df_clin = df_clin.merge(omics, on="slide_id", how="left", validate="one_to_one")
    df_clin = cross_validate(df_clin, n_splits=n_splits, n_bins=n_bins)
    return df_clin

In [67]:
def preload_wsi_features(df: pd.DataFrame, wsi_dir: Path) -> dict:
    """Lädt alle eindeutigen WSI-Feature-Dateien einmalig in den RAM."""
    wsi_dir = Path(wsi_dir)
    unique_slides = df['slide_id'].unique()
    cache = {}
    
    print(f"Lade {len(unique_slides)} WSI-Feature-Dateien in den RAM...")
    for slide_id in unique_slides:
        h5_path = wsi_dir / f"{slide_id}.h5"
        with h5py.File(h5_path, 'r') as f:
            # Direkt als float32-Tensor im RAM ablegen
            cache[slide_id] = torch.from_numpy(f['features'][:]).to(device, dtype=torch.float32) #device laedt die Daten auf die GPU, falls device='cuda', sonst auf die CPU
            
    ram_mb = sum(t.element_size() * t.nelement() for t in cache.values()) / (1024 ** 2)
    print(f"Fertig! Belegter RAM durch Feature-Cache: {ram_mb:.1f} MB")
    return cache


def scale_omics(train_df: pd.DataFrame, val_df: pd.DataFrame, omic_groups: list):
    """StandardScaler wird NUR auf dem Trainings-Fold gefittet (kein Leakage) und auf Train und Val angewendet.
    Gibt pro Signatur-Gruppe ein float32-Array [n_samples, n_genes_der_gruppe] zurueck."""
    all_cols = sorted({c for g in omic_groups for c in g})  # eindeutig: Gene koennen in mehreren Signaturen vorkommen
    scaler = StandardScaler().fit(train_df[all_cols].fillna(0).values)
    out = []
    for df in (train_df, val_df):
        scaled = pd.DataFrame(scaler.transform(df[all_cols].fillna(0).values), columns=all_cols, index=df.index)
        out.append([scaled[g].values.astype(np.float32) for g in omic_groups])
    return out[0], out[1]


class SurvivalWSIDataset(Dataset):
    def __init__(self, df: pd.DataFrame, wsi_dir: Path, omic_arrays: list, max_patches: int = None, feature_cache: dict = None):
        self.df = df.reset_index(drop=True)
        self.wsi_dir = Path(wsi_dir)
        self.max_patches = max_patches
        self.feature_cache = feature_cache
        # Pro-Sample-Daten einmalig vorbereiten (df.iloc im __getitem__ ist langsam)
        self.slide_ids = self.df['slide_id'].tolist()
        self.omics = [torch.from_numpy(a) for a in omic_arrays]   # 6 Tensoren [n_samples, d_i]
        self.targets = torch.tensor(self.df[['time_bin', 'censorship']].values, dtype=torch.long)
        self.surv = torch.tensor(self.df['survival_months'].values, dtype=torch.float32)
        self.cens = torch.tensor(self.df['censorship'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        slide_id = self.slide_ids[idx]
        
        # 1. Aus dem RAM-Cache lesen (falls vorhanden), sonst Fallback auf SSD
        if self.feature_cache is not None and slide_id in self.feature_cache:
            features_tensor = self.feature_cache[slide_id]
        else:
            h5_path = self.wsi_dir / f"{slide_id}.h5"
            with h5py.File(h5_path, 'r') as f:
                features_tensor = torch.from_numpy(f['features'][:]).to(torch.float32)

        # 2. Optionales Subsampling direkt mit PyTorch auf dem Tensor
        if self.max_patches is not None and features_tensor.size(0) > self.max_patches:
            indices = torch.randperm(features_tensor.size(0))[:self.max_patches]
            features_tensor = features_tensor[indices]
        
        omics = [o[idx] for o in self.omics]
        return features_tensor, omics, self.targets[idx], self.surv[idx], self.cens[idx]

In [68]:
from sksurv.metrics import concordance_index_censored

class NLLSurvLoss(nn.Module):
    """
    Discrete-Time Survival Loss (Negative Log-Likelihood).
    """
    def __init__(self, alpha: float = 0.15):
        super().__init__()
        self.alpha = alpha

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # target: [B, 2] -> Spalte 0 = time_bin, Spalte 1 = censorship (1 = zensiert, 0 = Event)
        y = target[:, 0].view(-1, 1)
        c = target[:, 1].view(-1, 1).float()

        hazards = torch.sigmoid(logits)
        S = torch.cumprod(1 - hazards, dim=1) # kumulatives Produkt: Tensor, bei dem jeder Eintrag die Wahrscheinlichkeit angibt, bis zu diesem spezifischen Zeitpunkt (und allen vorherigen) überlebt zu haben. 
        S_padded = torch.cat([torch.ones_like(c), S], dim=1)

        s_prev = torch.gather(S_padded, dim=1, index=y).clamp(min=1e-7)
        h_this = torch.gather(hazards, dim=1, index=y).clamp(min=1e-7)
        s_this = torch.gather(S_padded, dim=1, index=y + 1).clamp(min=1e-7)

        uncensored_loss = -(1 - c) * (torch.log(s_prev) + torch.log(h_this))
        censored_loss = -c * torch.log(s_this)

        loss = (1.0 - self.alpha) * censored_loss + uncensored_loss
        return loss.mean()


In [69]:
# Original-MCAT-Modell (Chen et al., ICCV 2021) aus dem lokalen Repo (angepasst: konfigurierbare WSI-Dimension, torch 2)
MCAT_REPO = "/loctmp/mue37538/models/MCAT/MCAT"
if MCAT_REPO not in sys.path:
    sys.path.insert(0, MCAT_REPO)
from models.model_coattn import MCAT_Surv


class MCAT(nn.Module):
    """Wrapper um MCAT_Surv mit der Schnittstelle forward(h, omics, loss_fn, label) -> (results_dict, None).
    h: [1, N, in_dim] WSI-Patches, omics: Liste aus 6 Tensoren [1, d_i] (eine pro Signatur). Batchgroesse 1."""
    def __init__(self, omic_sizes, in_dim=768, num_classes=4, fusion="concat",
                 model_size_wsi="small", model_size_omic="small", dropout=0.25):
        super().__init__()
        self.model = MCAT_Surv(fusion=fusion, omic_sizes=omic_sizes, n_classes=num_classes,
                               model_size_wsi=model_size_wsi, model_size_omic=model_size_omic,
                               dropout=dropout, wsi_dim=in_dim)
        # MCAT gibt nur hazards = sigmoid(logits) zurueck -> Logits per Hook am Klassifikator abgreifen
        self._logits = None
        self.model.classifier.register_forward_hook(lambda m, i, o: setattr(self, "_logits", o.reshape(1, -1)))

    def forward(self, h, omics, loss_fn=None, label=None):
        omics = [o.squeeze(0).float() for o in omics]
        self.model(x_path=h.squeeze(0).float(),
                   **{f"x_omic{i+1}": o for i, o in enumerate(omics)})
        logits = self._logits  # [1, num_classes]
        results_dict = {"logits": logits}
        if loss_fn is not None and label is not None:
            results_dict["loss"] = loss_fn(logits, label)
        return results_dict, None

In [70]:
def train_one_epoch(model, loader, loss_fn, optimizer, device, grad_accum_steps=8):
    model.train()
    total_loss = torch.zeros((), device=device)
    optimizer.zero_grad()
    num_batches = len(loader)

    for step, (features, omics, target, _, _) in enumerate(loader):
        features = features.to(device)  # [1, M, 768]
        omics = [o.to(device) for o in omics]  # 6 x [1, d_i]
        target = target.to(device)      # [1, 2]

        rem_batches = num_batches - (step // grad_accum_steps) * grad_accum_steps
        actual_accum_steps = min(grad_accum_steps, rem_batches)

        results_dict, _ = model(h=features, omics=omics, loss_fn=loss_fn, label=target)
        loss = results_dict['loss'] / actual_accum_steps
        loss.backward()

        total_loss += results_dict['loss'].detach() # Nur für Logging; kein .item() hier, das würde pro Schritt die GPU synchronisieren

        if (step + 1) % grad_accum_steps == 0 or (step + 1) == num_batches:
            optimizer.step()
            optimizer.zero_grad()

    return total_loss.item() / len(loader)


@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = torch.zeros((), device=device)
    all_risk_scores = []
    all_censorships = []
    all_event_times = []

    for features, omics, target, surv_months, censorship in loader:
        features = features.to(device)
        omics = [o.to(device) for o in omics]
        target = target.to(device)

        results_dict, _ = model(h=features, omics=omics, loss_fn=loss_fn, label=target)
        total_loss += results_dict['loss'].detach()

        # Risk Score aus den 4 Survival-Bins berechnen
        hazards = torch.sigmoid(results_dict['logits'])
        survival = torch.cumprod(1 - hazards, dim=1) #Berechnet die kumulative Überlebenswahrscheinlichkeit über die Zeit, indem das Produkt von 1 - hazards gebildet wird.
        risk = -torch.sum(survival, dim=1) # Berechnet den Risikowert als negative Summe der Überlebenswahrscheinlichkeiten (höherer Wert = höheres Risiko)


        all_risk_scores.append(risk.detach())  # bleibt auf der GPU, ein Transfer am Ende
        all_censorships.extend(censorship.numpy())
        all_event_times.extend(surv_months.numpy())

    all_risk_scores = torch.cat(all_risk_scores).cpu().numpy()

    # sksurv erwartet event_indicator = True für Todesfall (1 - censorship)
    event_indicators = (1 - np.array(all_censorships)).astype(bool)
    c_index = concordance_index_censored(
        event_indicators,
        np.array(all_event_times),
        np.array(all_risk_scores)
    )[0]

    return total_loss.item() / len(loader), c_index, all_risk_scores

In [71]:
import json
from datetime import datetime
from pathlib import Path
import pandas as pd
import numpy as np

class ExperimentLogger:
    def __init__(self, base_dir: str, config: dict):
        self.config = config
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        run_name = f"{config['cohort']}_{config['model_type']}_{timestamp}"
        
        self.base_dir = Path(base_dir)
        self.run_dir = self.base_dir / run_name
        self.ckpt_dir = self.run_dir / "checkpoints"
        self.ckpt_dir.mkdir(parents=True, exist_ok=True)
        
        # 1. Config als JSON speichern
        with open(self.run_dir / "config.json", "w") as f:
            json.dump(self.config, f, indent=4)
            
        self.epoch_logs = []
        self.oof_records = []

    def log_splits(self, df: pd.DataFrame):
        # 2. Exakte Splits und Bins sichern
        cols = ["case_id", "slide_id", "site", "survival_months", "censorship", "event", "time_bin", "fold"]
        available_cols = [c for c in cols if c in df.columns]
        df[available_cols].to_csv(self.run_dir / "dataset_splits.csv", index=False)

    def log_epoch(self, fold: int, epoch: int, train_loss: float, val_loss: float, val_c_index: float):
        # 3. Verlauf pro Epoche merken
        self.epoch_logs.append({
            "fold": fold,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_c_index": val_c_index
        })

    def log_oof_predictions(self, val_df: pd.DataFrame, risk_scores: list, fold: int):
        # 4. Beste Validierungs-Vorhersagen dieses Folds für spätere Kaplan-Meier-Analysen speichern
        oof_df = val_df[["case_id", "slide_id", "survival_months", "censorship", "event", "time_bin"]].copy()
        oof_df["fold"] = fold
        oof_df["risk_score"] = risk_scores
        self.oof_records.append(oof_df)

    def finalize(self, fold_c_best: list, fold_c_loss: list = None, fold_c_last: list = None):
        pd.DataFrame(self.epoch_logs).to_csv(self.run_dir / "metrics_per_epoch.csv", index=False)
        if self.oof_records:
            pd.concat(self.oof_records, ignore_index=True).to_csv(self.run_dir / "oof_predictions.csv", index=False)
        
        summary = {
            "run_name": self.run_dir.name,
            "mean_c_index": round(float(np.mean(fold_c_best)), 4),
            "std_c_index": round(float(np.std(fold_c_best)), 4),
            "fold_c_indices": [round(float(x), 4) for x in fold_c_best],
        }
        if fold_c_loss is not None:
            summary["mean_c_at_best_loss"] = round(float(np.mean(fold_c_loss)), 4)
            summary["std_c_at_best_loss"] = round(float(np.std(fold_c_loss)), 4)
        if fold_c_last is not None:
            summary["mean_c_last_epoch"] = round(float(np.mean(fold_c_last)), 4)
            summary["std_c_last_epoch"] = round(float(np.std(fold_c_last)), 4)

        summary.update(self.config)
        
        with open(self.run_dir / "summary.json", "w") as f:
            json.dump(summary, f, indent=4)
            
        master_csv = self.base_dir / "experiment_summary.csv"
        summary_flat = {k: str(v) if isinstance(v, list) else v for k, v in summary.items()}
        df_row = pd.DataFrame([summary_flat])
        df_row.to_csv(master_csv, mode="a", header=not master_csv.exists(), index=False)
        
        print(f"\nAlle Ergebnisse archiviert unter: {self.run_dir}")

In [72]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Nutze Device: {device}")

Nutze Device: cuda


In [73]:
import random
# 1. Zentrale Konfiguration definieren

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

config = {
    "cohort": "TCGA-KIRP",
    "model_type": "mcat",
    "in_dim": 768, # equal to CONCH v1.5 embedding dimension
    "fusion": "concat",  # 'concat' oder 'bilinear'
    "model_size_wsi": "small",
    "model_size_omic": "small",
    "num_classes": 4,
    "epochs": 20,
    "lr": 2e-4,
    "weight_decay": 1e-5,
    "grad_accum_steps": 32, # Wie in Papern Mcat, Motcat
    "max_patches_train": None,
    "loss_alpha": 0.15,
    "seed": 85,
    "n_jobs": 5   # Folds parallel in eigenen Prozessen (1 = nacheinander, mit Feature-Cache auf der GPU)
}

seed_everything(config["seed"])

dataset = build_dataset(BASE_DIR / config["cohort"], n_bins=config["num_classes"], n_splits=5)

# 6 Signatur-Gruppen (MCAT): Spaltennamen der Omics-Features je Gruppe
omic_groups = get_omic_groups(dataset.columns, BASE_DIR / "shared/gene_sets/signatures.csv")
omic_sizes = [len(g) for g in omic_groups]
print("Omics-Dimensionen pro Signatur:", omic_sizes)
config["omic_sizes"] = omic_sizes

logger = ExperimentLogger(base_dir="./runs", config=config)
logger.log_splits(dataset)
WSI_DIR = BASE_DIR / config["cohort"] / "wsi_features/conch_v15"


Fold-Verteilung (Größe | Events):
  Fold 0: n= 54 | events= 9 | sites=6
  Fold 1: n= 48 | events= 8 | sites=6
  Fold 2: n= 54 | events= 7 | sites=7
  Fold 3: n= 47 | events= 9 | sites=5
  Fold 4: n= 56 | events= 7 | sites=7
Omics-Dimensionen pro Signatur: [83, 306, 496, 415, 135, 424]


In [74]:
from joblib import Parallel, delayed


def _fold_log(msg, log_path, echo):
    with open(log_path, "a") as f:
        f.write(msg + "\n")
    if echo:
        print(msg)


def run_fold(fold, dataset, config, omic_groups, wsi_dir, ckpt_dir, log_path, feature_cache=None, device_str="cuda"):
    """Trainiert und validiert einen Fold. Laeuft im Hauptprozess (n_jobs=1) oder in einem eigenen Prozess (n_jobs>1).
    Ausgaben gehen in log_path. Gibt nur Ergebnisse zurueck."""
    echo = feature_cache is not None  # sequenziell -> zusaetzlich im Notebook ausgeben
    dev = torch.device(device_str)
    torch.set_num_threads(max(1, os.cpu_count() // max(1, config["n_jobs"])))  # CPU nicht ueberbuchen
    seed_everything(config["seed"] + fold)
    _fold_log(f"==================== FOLD {fold} ====================", log_path, echo)

    train_df = dataset[dataset['fold'] != fold]
    val_df = dataset[dataset['fold'] == fold]
    train_df, val_df = compute_fold_time_bins(train_df, val_df, n_bins=config["num_classes"])

    # Omics: Scaler nur auf dem Trainings-Fold fitten
    train_omics, val_omics = scale_omics(train_df, val_df, omic_groups)

    train_ds = SurvivalWSIDataset(train_df, wsi_dir=wsi_dir, omic_arrays=train_omics,
                                  max_patches=config["max_patches_train"], feature_cache=feature_cache)
    val_ds = SurvivalWSIDataset(val_df, wsi_dir=wsi_dir, omic_arrays=val_omics,
                                max_patches=None, feature_cache=feature_cache)
    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0, pin_memory=False)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0, pin_memory=False)

    model = MCAT(
        omic_sizes=[len(g) for g in omic_groups],
        in_dim=config["in_dim"],
        num_classes=config["num_classes"],
        fusion=config["fusion"],
        model_size_wsi=config["model_size_wsi"],
        model_size_omic=config["model_size_omic"],
    ).to(dev)

    loss_fn = NLLSurvLoss(alpha=config["loss_alpha"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"])

    best_c_index = -1.0
    best_val_loss = float("inf")
    c_index_at_best_loss = 0.0
    best_risk_scores = None
    epoch_logs = []

    for epoch in range(config["epochs"]):
        train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer, dev, grad_accum_steps=config["grad_accum_steps"])
        val_loss, val_c_index, val_risk_scores = evaluate(model, val_loader, loss_fn, dev)
        scheduler.step()

        epoch_logs.append((fold, epoch + 1, train_loss, val_loss, val_c_index))
        _fold_log(f"[Fold {fold}] Epoch {epoch+1:02d}/{config['epochs']:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val C-Index: {val_c_index:.4f}", log_path, echo)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            c_index_at_best_loss = val_c_index

        if val_c_index > best_c_index:
            best_c_index = val_c_index
            best_risk_scores = val_risk_scores
            torch.save(model.state_dict(), Path(ckpt_dir) / f"best_model_fold_{fold}.pt")

    _fold_log(f"--> Fold {fold} | Best C-Index: {best_c_index:.4f} | C-Index @ Best Loss: {c_index_at_best_loss:.4f} | Last C-Index: {val_c_index:.4f}", log_path, echo)
    return {"fold": fold, "epoch_logs": epoch_logs, "best_c": best_c_index,
            "c_at_best_loss": c_index_at_best_loss, "last_c": val_c_index, "best_risk": best_risk_scores}

In [75]:
n_jobs = config["n_jobs"]
n_folds = 5

# Sequenziell: alle Features einmalig auf die GPU. Parallel: jeder Prozess liest die h5-Dateien selbst
# (geteilter OS-Seitencache), damit der VRAM nicht n_jobs-fach belegt wird.
wsi_cache = preload_wsi_features(dataset, WSI_DIR) if n_jobs == 1 else None

# Worker-Prozesse muessen das MCAT-Repo importieren koennen
os.environ["PYTHONPATH"] = MCAT_REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ["PYTHONWARNINGS"] = "ignore:enable_nested_tensor is True"  # Warnung auch in den Worker-Prozessen unterdruecken


import threading, time

def _tail_logs(log_files, stop_event, interval=2.0):
    """Druckt neue Zeilen der fold_*.log live ins Notebook (Worker haben kein eigenes stdout)."""
    pos = {f: 0 for f in log_files}
    while True:
        done = stop_event.is_set()  # nach dem Stop noch einmal vollstaendig lesen
        for f in log_files:
            if f.exists():
                with open(f) as fh:
                    fh.seek(pos[f])
                    for line in fh:
                        if line.startswith("[Fold") or line.startswith("-->"):
                            print(line.rstrip())
                    pos[f] = fh.tell()
        if done:
            return
        time.sleep(interval)

if n_jobs > 1:
    print(f"Starte {n_folds} Folds mit {n_jobs} parallelen Prozessen. Fortschritt: tail -f {logger.run_dir}/fold_*.log")

log_files = [logger.run_dir / f"fold_{fold}.log" for fold in range(n_folds)]
stop_tail = threading.Event()
tail_thread = None
if n_jobs > 1:
    tail_thread = threading.Thread(target=_tail_logs, args=(log_files, stop_tail), daemon=True)
    tail_thread.start()

try:
    results = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(run_fold)(fold, dataset, config, omic_groups, WSI_DIR, logger.ckpt_dir,
                          log_files[fold], wsi_cache, str(device))
        for fold in range(n_folds)
    )
finally:
    stop_tail.set()
    if tail_thread is not None:
        tail_thread.join()
results = sorted(results, key=lambda r: r["fold"])

fold_results_best_c, fold_results_best_loss, fold_results_last = [], [], []
for r in results:
    for (f, ep, tl, vl, vc) in r["epoch_logs"]:
        logger.log_epoch(f, ep, tl, vl, vc)
    # val_df mit denselben Fold-Bins wie im Training fuer die OOF-Vorhersagen
    _, val_df = compute_fold_time_bins(dataset[dataset['fold'] != r["fold"]], dataset[dataset['fold'] == r["fold"]],
                                       n_bins=config["num_classes"])
    logger.log_oof_predictions(val_df, r["best_risk"], r["fold"])
    fold_results_best_c.append(r["best_c"])
    fold_results_best_loss.append(r["c_at_best_loss"])
    fold_results_last.append(r["last_c"])
    print(f"Fold {r['fold']} | Best C-Index: {r['best_c']:.4f} | @ Best Loss: {r['c_at_best_loss']:.4f} | Last: {r['last_c']:.4f}")

# GESAMTERGEBNIS & ARCHIVIERUNG
print("\n==================================================")
print(f"5-Fold CV ({config['cohort']}) - Best Val C-Index : {np.mean(fold_results_best_c):.4f} ± {np.std(fold_results_best_c):.4f}")
print(f"5-Fold CV ({config['cohort']}) - @ Best Val Loss  : {np.mean(fold_results_best_loss):.4f} ± {np.std(fold_results_best_loss):.4f}")
print(f"5-Fold CV ({config['cohort']}) - Last Epoch       : {np.mean(fold_results_last):.4f} ± {np.std(fold_results_last):.4f}")
logger.finalize(fold_results_best_c, fold_results_best_loss, fold_results_last)

Starte 5 Folds mit 5 parallelen Prozessen. Fortschritt: tail -f runs/TCGA-KIRP_mcat_20261119_172732/fold_*.log


[Fold 0] Epoch 01/20 | Train Loss: 1.6092 | Val Loss: 1.1336 | Val C-Index: 0.8221
[Fold 1] Epoch 01/20 | Train Loss: 1.4305 | Val Loss: 1.3718 | Val C-Index: 0.5957
[Fold 2] Epoch 01/20 | Train Loss: 1.4554 | Val Loss: 1.1845 | Val C-Index: 0.7081
[Fold 3] Epoch 01/20 | Train Loss: 1.4291 | Val Loss: 1.0962 | Val C-Index: 0.6486
[Fold 4] Epoch 01/20 | Train Loss: 1.4137 | Val Loss: 1.2227 | Val C-Index: 0.8433
[Fold 0] Epoch 02/20 | Train Loss: 1.1720 | Val Loss: 0.6584 | Val C-Index: 0.8773
[Fold 1] Epoch 02/20 | Train Loss: 0.9662 | Val Loss: 0.7624 | Val C-Index: 0.6170
[Fold 2] Epoch 02/20 | Train Loss: 0.9964 | Val Loss: 0.6567 | Val C-Index: 0.7177
[Fold 3] Epoch 02/20 | Train Loss: 0.9947 | Val Loss: 0.7245 | Val C-Index: 0.8108
[Fold 4] Epoch 02/20 | Train Loss: 1.0157 | Val Loss: 0.7955 | Val C-Index: 0.8731
[Fold 0] Epoch 03/20 | Train Loss: 0.7155 | Val Loss: 0.4831 | Val C-Index: 0.9202
[Fold 1] Epoch 03/20 | Train Loss: 0.5688 | Val Loss: 0.6313 | Val C-Index: 0.6340
[Fol

In [76]:
torch.cuda.empty_cache()